# AI Interview Coach - Preference Scoring (RLAIF)

This notebook runs the judge model to compare pairs of feedback responses and determine which one is better. This is used to build the preference dataset for RLAIF/DPO training.

## 1. Mount Google Drive
Mounting Drive ensures that your results (`.jsonl` files) are saved permanently even if the Colab runtime is disconnected.

In [1]:
from google.colab import drive
drive.mount('/content/drive')

# Update this path to the location of your project in Google Drive
PROJECT_PATH = "/content/drive/MyDrive/ai-interview-coach"

import os
if not os.path.exists(PROJECT_PATH):
    print(f"Creating project directory: {PROJECT_PATH}")
    !git clone https://github.com/dcyforjob2020/ai-interview-coach.git "{PROJECT_PATH}"

%cd "{PROJECT_PATH}"

# Force a clean state before pulling
!git reset --hard
!git clean -fdx

# Pull latest changes
!git pull origin main

Mounted at /content/drive
/content/drive/MyDrive/ai-interview-coach
Updating files: 100% (34/34), done.
HEAD is now at 322654d Fix: Resolve torchvision compatibility and add restart instructions
From https://github.com/dcyforjob2020/ai-interview-coach
 * branch            main       -> FETCH_HEAD
Already up to date.


## 2. Setup Environment
Install the necessary libraries for running large language models and quantization.

In [2]:
!pip install -r requirements.txt

print("\n" + "="*60)
print("SETUP COMPLETE. YOU MUST RESTART THE RUNTIME NOW")
print("(Go to 'Runtime' -> 'Restart session' in the menu bar)")
print("="*60)

  Cloning https://github.com/huggingface/transformers.git to /tmp/pip-req-build-_ttdqegf
  Running command git clone --filter=blob:none --quiet https://github.com/huggingface/transformers.git /tmp/pip-req-build-_ttdqegf
  Resolved https://github.com/huggingface/transformers.git to commit 39603d0e5cdb6f00e8d473d7fcbb01032d709181
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 41.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 761.1/761.1 kB 61.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 529.0/529.0 kB 48.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.9/48.9 MB 55.9 MB/s eta 0:00:00
  Created wheel for transformers: filename=transformers-5.10.0.dev0-py3-none-any.whl size=11993530 sha256=2c3d21ca62712e7a73a342d3468f60ec7a33ac18c4a968f02e1cea171bc2697d
  Stored in directory: /tmp/pip-ephem-wheel-

## 3. Configuration
Adjust the paths and model as needed.
- `INPUT_PATH`: The file containing candidates to be compared. (e.g., `train/preference_candidates.jsonl` for all data)
- `OUTPUT_PATH`: Where the resulting preference pairs will be saved.
- `MODEL_NAME`: The judge model to use (default: `Qwen/Qwen3.5-9B`).
- `QUANTIZE`: Enable 4-bit quantization to fit the model on consumer GPUs.

In [3]:
# List available candidate files
!ls train/*.jsonl

INPUT_PATH = "train/preference_candidates.jsonl"  # Set to the file containing your data
OUTPUT_PATH = "train/preference_pairs.jsonl"       # Where to save the results
MODEL_NAME = "Qwen/Qwen3.5-9B"
QUANTIZE = True
LIMIT = None
OVERWRITE = False

train/preference_candidates_1.jsonl  train/preference_candidates_4.jsonl
train/preference_candidates_2.jsonl  train/preference_candidates.jsonl
train/preference_candidates_3.jsonl  train/preference_candidates_original.jsonl


## 4. Run Preference Scoring (Whole Dataset)
Run this cell to process the entire input file in one go.

In [4]:
limit_arg = f"--limit {LIMIT}" if LIMIT else ""
quantize_arg = "--quantize" if QUANTIZE else ""
overwrite_arg = "--overwrite" if OVERWRITE else ""

!python eval/score_preferences.py \
    --input {INPUT_PATH} \
    --output {OUTPUT_PATH} \
    --model {MODEL_NAME} \
    {limit_arg} \
    {quantize_arg} \
    {overwrite_arg}

Loading judge model: Qwen/Qwen3.5-9B (quantize=True)
config.json: 100% 3.13k/3.13k [00:00<00:00, 9.65MB/s]
tokenizer_config.json: 100% 16.7k/16.7k [00:00<00:00, 29.5MB/s]
vocab.json: 100% 6.72M/6.72M [00:00<00:00, 80.6MB/s]
merges.txt: 100% 3.35M/3.35M [00:00<00:00, 120MB/s]
tokenizer.json: 100% 12.8M/12.8M [00:00<00:00, 27.9MB/s]
chat_template.jinja: 100% 7.76k/7.76k [00:00<00:00, 20.6MB/s]
model.safetensors.index.json: 100% 79.7k/79.7k [00:00<00:00, 143MB/s]
Fetching 4 files: 100% 4/4 [00:53<00:00, 13.30s/it]
Download complete: 100% 19.3G/19.3G [00:53<00:00, 387MB/s]                [transformers] The fast path is not available because one of the required library is not installed. Falling back to torch implementation. To install follow https://github.com/fla-org/flash-linear-attention#installation and https://github.com/Dao-AILab/causal-conv1d
Download complete: 100% 19.3G/19.3G [00:53<00:00, 362MB/s]
Loading weights:   0% 2/427 [00:01<03:49,  1.85it/s]/usr/local/lib/python3.12/dist-p